# 1. A tour of the MCP server

We'll do what an agent does: look at the tools, start a poster job, wait for it, look at the result.

Making a poster takes a few seconds, so there's no single "make and wait" tool. `create_poster_job` returns a job id straight away, and `get_job_status` is polled until the job is done.

```mermaid
stateDiagram-v2
    [*] --> processing: create_poster_job
    processing --> completed
    processing --> failed
    processing --> cancelled: cancel_job
```

In [1]:
from demoUtils import connectMcp, callTool, show, waitForJob, showImages

## the tools

An agent starts by asking what tools exist. The name, description and input schema are all it knows about each one.

In [2]:
async with connectMcp() as session:
    tools = (await session.list_tools()).tools

for tool in tools:
    required = tool.inputSchema.get("required", [])
    print(f"{tool.name}   (required: {', '.join(required) or 'none'})")
    print("   ", tool.description.split(". ")[0] + ".")
    print()

list_templates   (required: none)
    Browse ready-made poster templates.

create_poster_job   (required: product_image, background_image, prompt)
    Start generating poster images.

get_job_status   (required: job_id)
    Check a poster job.

cancel_job   (required: job_id)
    Stop a job that is still processing, e.g.



The limits (1 to 4 images, 500 characters, `1k` or `2k`) are in the schema itself, so a model can see them before it makes a mistake:

In [3]:
createTool = next(t for t in tools if t.name == "create_poster_job")
for name, spec in createTool.inputSchema["properties"].items():
    limits = {k: spec[k] for k in ("minimum", "maximum", "maxLength", "enum") if k in spec}
    print(f"{name:17} {limits if limits else ''}")
    print(f"    {spec.get('description', '')}")

product_image     
    The product photo: a public https:// URL or an asset:// reference.
background_image  
    The background / scene image: a public https:// URL or an asset:// reference.
prompt            {'maxLength': 500}
    What the poster should show and say (language, offer, mood). Maximum 500 characters.
template_id       
    Optional template id from list_templates. Omit for a free-form poster.
image_count       {'minimum': 1, 'maximum': 4}
    How many poster variations to generate, 1 to 4. Costs scale with this number; default 1.
resolution        {'enum': ['1k', '2k']}
    "1k" (default, fast) or "2k" (higher quality, use only when the user asks for print/large format).
idempotency_key   
    Optional. Reuse the SAME key when retrying after a timeout or error so no duplicate job (and no extra cost) is created. If omitted, one is derived from the inputs, so repeating identical inputs returns the existing job. Pass a NEW key to deliberately get another batch with identica

## templates

In [4]:
async with connectMcp() as session:
    sale = await callTool(session, "list_templates", category="sale")
    coffee = await callTool(session, "list_templates", search="coffee")

print("category = sale:")
show(sale["data"]["templates"])
print("search = coffee:")
show(coffee["data"]["templates"])

category = sale:
[
  {
    "id": "tpl_sale_bold",
    "name": "Weekend Sale Bold",
    "category": "sale"
  },
  {
    "id": "tpl_sale_clean",
    "name": "Clean Discount Banner",
    "category": "sale"
  }
]
search = coffee:
[
  {
    "id": "tpl_coffee_steam",
    "name": "Coffee Steam",
    "category": "food-and-beverage"
  }
]


## start a job

Images are `https://` urls (or `asset://` references). The prompt can be Arabic. The result has a `next_step` hint for the agent, and an `idempotency_key` it can reuse to retry safely.

In [5]:
posterArgs = dict(
    product_image="https://cdn.example.com/latte.png",
    background_image="https://cdn.example.com/cafe.png",
    prompt="عرض نهاية الأسبوع: خصم 20% على جميع أنواع اللاتيه",
    template_id="tpl_sale_bold",
    image_count=2,
)

async with connectMcp() as session:
    created = await callTool(session, "create_poster_job", **posterArgs)

show(created["data"])
jobId = created["data"]["job_id"]

{
  "job_id": "f259c2820966",
  "status": "processing",
  "idempotency_key": "auto-a77267dfe98b0c0b6834a572a0c7a5e2",
  "duplicate": false,
  "next_step": "Call get_job_status with job_id=f259c2820966 in about 5 seconds."
}


## wait for it

About 1 in 10 jobs fails on purpose. If yours does, run the cell above again. Notebook 2 shows how to handle it properly.

In [6]:
async with connectMcp() as session:
    final = await waitForJob(session, jobId)

if final["data"]["status"] == "completed":
    showImages(final["data"]["images"])
else:
    show(final["data"])

poll 1: processing (elapsed 2.7s)
poll 2: completed (elapsed 4.2s)


Images come back as links, not image data. Links are cheap for a model, they expire after 15 minutes, and calling `get_job_status` again gives fresh ones.

In [7]:
show(final["data"])

{
  "job_id": "f259c2820966",
  "status": "completed",
  "elapsed_seconds": 4.2,
  "links_expire_in_seconds": 900,
  "images": [
    {
      "index": 1,
      "resolution": "1k",
      "url": "http://localhost:8000/images/f259c2820966/1.svg?exp=1792178443&sig=fb8fd12bdcb47e15be9f86be5d50d186"
    },
    {
      "index": 2,
      "resolution": "1k",
      "url": "http://localhost:8000/images/f259c2820966/2.svg?exp=1792178443&sig=69200372dae17108e310609f72e777fe"
    }
  ],
  "next_step": "Give the image urls to the user. Do not call this tool again unless links expired."
}


## cancel a job

In [8]:
async with connectMcp() as session:
    other = await callTool(session, "create_poster_job", **{**posterArgs, "prompt": "A poster we will cancel"})
    cancelled = await callTool(session, "cancel_job", job_id=other["data"]["job_id"])
    after = await callTool(session, "get_job_status", job_id=other["data"]["job_id"])

show(cancelled["data"])
print("status afterwards:", after["data"]["status"])

{
  "job_id": "5f1530c69279",
  "status": "cancelled",
  "elapsed_seconds": 0.0,
  "next_step": "Job cancelled. Create a new job if the user still wants posters."
}
status afterwards: cancelled


Next: what happens when things go wrong.